# Phase 6: Model Training

This notebook trains the baseline valuation model on `log1p(value_per_area)`.

Scope:
- load the Phase 5 model training dataset
- prepare numeric and categorical features
- split train/test reproducibly
- train a RandomForest baseline
- save the trained model and training summary


In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import geopandas as gpd

from backend.src.config import configure_logging, ensure_directories, settings
from backend.src.model_training import save_model_artifacts, train_random_forest_model

configure_logging()
ensure_directories()
PROJECT_ROOT, settings


(WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc'),
 Settings(project_root=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc'), raw_data_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/data'), interim_data_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/data/interim'), processed_data_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/data/processed'), reports_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/reports'), models_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/models'), transaction_file='tran_data.xlsx', property_shapefile='ai_usecase_data240326.shp', road_shapefile='ai_usecase_road240326.shp', facility_shapefile='ai_usecase_facilities240326.shp', log_level='INFO'))

In [2]:
training_gdf = gpd.read_parquet(settings.processed_data_dir / 'model_training_dataset.parquet')
training_gdf.shape


(273587, 34)

In [3]:
artifacts = train_random_forest_model(training_gdf)
artifacts.summary


c:\Users\pulkit verma\miniconda3\envs\xcel\lib\site-packages\sklearn\impute\_base.py:590: FutureWarning: Currently, when `keep_empty_feature=False` and `strategy="constant"`, empty features are not dropped. This behaviour will change in version 1.8. Set `keep_empty_feature=True` to preserve this behaviour.
  warnings.warn(


ModelTrainingSummary(input_row_count=273587, rows_used_for_training=273587, train_row_count=218869, test_row_count=54718, train_row_count_after_sampling=10000, target_column='value_per_area', target_transform='log1p(value_per_area)', model_name='RandomForestRegressor', numeric_features=['Area', 'Approach_Road_Width', 'distance_to_nearest_road', 'distance_to_nearest_facility', 'facility_count_500m', 'facility_count_1km', 'latitude', 'longitude'], categorical_features=['property_district_Name', 'Mouza_Name', 'Zone_no', 'Proposed_Land_use_Name', 'Nature_Land_use_Name', 'Urban', 'Rural', 'Road_Category', 'Flat_or_Land', 'Litigated_Property'], random_state=42, test_size=0.2, n_estimators=20, max_depth=14, min_samples_leaf=20, max_training_sample_size=10000)

In [4]:
artifacts.X_train.head()


,Area,Approach_Road_Width,distance_to_nearest_road,distance_to_nearest_facility,facility_count_500m,facility_count_1km,latitude,longitude,property_district_Name,Mouza_Name,Zone_no,Proposed_Land_use_Name,Nature_Land_use_Name,Urban,Rural,Road_Category,Flat_or_Land,Litigated_Property
134979,959.000000,10.0,33.238198,503.683612,0.0,13.0,23.247843,87.874286,Purba Bardhaman,Nari,0,NaN,NaN,Yes,No,NaN,Flat,NO
151347,125.000000,20.0,NaN,NaN,0.0,0.0,NaN,NaN,Purba Bardhaman,Katrapota,0,NaN,NaN,No,Yes,NaN,Flat,NO
239709,0.310000,0.0,25.767059,239.246558,4.0,4.0,22.540302,88.146243,Howrah,Panchla,0,Pukur,Pukur,No,Yes,NaN,Land,NO
236674,13.390000,0.0,114.136579,225.101293,2.0,9.0,22.557259,88.168589,Howrah,Nabghara,0,Sali,Sali,No,Yes,NaN,Land,NO
269093,0.309375,6.0,16.190028,87.501028,5.0,22.0,22.581265,88.227283,Howrah,Mashila,0,Bastu,Bastu,No,Yes,NaN,Land,NO


In [5]:
model_path, summary_path = save_model_artifacts(artifacts, settings.models_dir, settings.reports_dir)
model_path, summary_path


2026-04-28 17:38:01,763 | INFO | src.model_training | Saved trained model to \\wsl.localhost\Ubuntu\home\pulkitv52\valuation_poc\models\valuation_model.pkl
2026-04-28 17:38:01,766 | INFO | src.model_training | Saved training summary to \\wsl.localhost\Ubuntu\home\pulkitv52\valuation_poc\reports\model_training_summary.json


(WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/models/valuation_model.pkl'),
 WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/reports/model_training_summary.json'))

In [6]:
json.loads(Path(summary_path).read_text())


{'input_row_count': 273587,
 'rows_used_for_training': 273587,
 'train_row_count': 218869,
 'test_row_count': 54718,
 'train_row_count_after_sampling': 10000,
 'target_column': 'value_per_area',
 'target_transform': 'log1p(value_per_area)',
 'model_name': 'RandomForestRegressor',
 'numeric_features': ['Area',
  'Approach_Road_Width',
  'distance_to_nearest_road',
  'distance_to_nearest_facility',
  'facility_count_500m',
  'facility_count_1km',
  'latitude',
  'longitude'],
 'categorical_features': ['property_district_Name',
  'Mouza_Name',
  'Zone_no',
  'Proposed_Land_use_Name',
  'Nature_Land_use_Name',
  'Urban',
  'Rural',
  'Road_Category',
  'Flat_or_Land',
  'Litigated_Property'],
 'random_state': 42,
 'test_size': 0.2,
 'n_estimators': 20,
 'max_depth': 14,
 'min_samples_leaf': 20,
 'max_training_sample_size': 10000}